# Forecasting GDP Growth
## Is accounting information useful for GDP forecast?
This project tasks you with forecasting quarterly nominal GDP growth rate in the United States. The dataset contains estimated nominal gross domestic product (GDP) annual growth rate in each quarter from 1990 to 2020, released by the Bureau of Economic Analysis (BEA) of the United States Department of Commerce. The estimate is the “third” or "final" estimate of GDP issued in the third month after the relevant quarter which is based on more complete source data than were available for the "second" estimate issued in the second month and the "advance" estimate issued in the first month after the relevant quarter. You may refer to the [BEA website](https://www.bea.gov/resources/learning-center/what-to-know-gdp) for more information on the estimate of GDP in the United States. You will train models to forecast the BEA's final GDP estimate for a quarter. Specifically you are required to answer this question: Is accounting information useful for GDP forecast? You will be allowed to use any outside data to build your model.

The evaluation pages describes how submissions will be scored and how students should format their submissions. --- The evaluation metric for this competition is [RMSE](https://en.wikipedia.org/wiki/Root-mean-square_deviation). RMSE is a measure of the accuracy of your predictions.

**Files**
- GDP_Forecast_train.csv - the training set
- GDP_Forecast_test.csv - the test set
- GDP_Forecast_sampleSubmission.csv - a sample submission file in the correct format

**Columns**
- YQ - The year and quarter variable. 1990Q1 refers to the first calendar quarter of 1990, ie, Jan - March 1990
- NGDP - nominal GDP growth rate in the quarter of the year. It is obtained from the BEA final estimate which is available at the BEA website.

**Features** (no features data provided)

Following Datar et al. (2020), you may consider the following data as your predictors/features:
- Accounting data - through COMPUSTAT on WRDS
- Stock price/return data - through CRSP on WRDS
- Analysts' forecast of GDP - through Survey of Professional Forecasters
- Any other data

**Additional Notes**
- Use RMSE to measure robustness to outliers (lower = better)
- Use Adjusted R-square to understand how much variance is explained (higher = better)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from sklearn.metrics import mean_squared_error

train = pd.read_csv("data/GDP_Forecast_train.csv")
test = pd.read_csv("data/GDP_Forecast_test.csv")

def create_submission(test_df, y_pred, filename):
    df = pd.DataFrame({"YQ": test_df["YQ"], "NGDP": y_pred})
    df.to_csv(f"attempts/{filename}.csv", index = False)

def calc_performance(formula1, model1, formula2, model2):
    print(pd.DataFrame({'model'   : [formula1, formula2],
                    'Adj.R^2' : [model1.rsquared_adj, model2.rsquared_adj]}))
    print()
    print(anova_lm(model1, model2, test = "Chisq"))

## splitting year-quarter to check for traits in yearly patterns vs quarterly pattern
# ensure year is an integer
train["year"] = train["YQ"].str[:4].astype(int)
test["year"] = test["YQ"].str[:4].astype(int)
# keep qtr as string / category
train["qtr"] = train["YQ"].str[4:]
test["qtr"] = test["YQ"].str[4:]
train.head()

,YQ,NGDP,year,qtr
0,1990Q1,9.0,1990,Q1
1,1990Q2,6.1,1990,Q2
2,1990Q3,3.7,1990,Q3
3,1990Q4,-0.7,1990,Q4
4,1991Q1,2.0,1991,Q1


### Base Model

In [2]:
# running base linear model
formula0 = "NGDP ~ year + qtr"
base_model = smf.ols(formula0, train).fit()
# printing summary of base_model
print(base_model.summary())

# get predicted values based off model
y_pred = base_model.predict(test)
y_true = test["NGDP"]
create_submission(test, y_pred, "0_base_forecast")
# evaluate using RMSE:
print(f"\nRMSE: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     3.324
Date:                Wed, 04 Mar 2026   Prob (F-statistic):             0.0134
Time:                        17:59:03   Log-Likelihood:                -244.91
No. Observations:                 104   AIC:                             499.8
Df Residuals:                      99   BIC:                             513.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    219.8404     68.424      3.213      0.0

Insights: year is statistically significant with a p-value of 0.002 while quarter is mostly not significant, with quarter 3 and 4 having p-values of 0.434 and 0.543 respectively.

It is also recognised that the condition number is large, suggesting strong multicollinearity or other numerical problems.

### How Accounting Data affects GDP (COMPUSTAT on WRDS)
[Compustat North America](https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/north-america-daily/) is a database of U.S. and Canadian fundamental and market information on active and inactive publicly held companies. It provides more than 300 annual and 100 quarterly Income Statement, Balance Sheet, Statement of Cash Flows, and supplemental data items. For most companies, annual history is available back to 1950 and quarterly history back to 1962 with monthly market history back to 1962.

- (gvkey) Global Company Key
- (conm) Company Name
- (revtq) The "Top Line" Signal: Revenue - Total
- (ppentq) The "Investment" Signal: Property Plant and Equipment - Total (Net)
- (invtq) The "Inventory" Signal: Inventories - Total
- (oibdpq) The "Operations" Signal: Operating Income Before Depreciation - Quarterly
- (xrdq) The "Innovation" Signal: Research and Development Expense

** To consider [Execucomp](https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/execucomp/): Executive compensation data collected directly from each company’s annual proxy (DEF14A SEC form)

In [ ]:
# Using Compustat North America: Fundamentals Quarterly
# https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/north-america-daily/

compustat = pd.read_csv("data/compustat_quarterly_financials2.csv")
## dropping columns with no unique data
compustat.drop(columns = ["curcdq", "datafmt", "indfmt", "consol"], inplace = True)
## shape of data
print(f"invtq: {compustat[compustat["invtq"].isna()].shape}")
print(f"oibdpq: {compustat[compustat["oibdpq"].isna()].shape}")
print(f"ppentq: {compustat[compustat["ppentq"].isna()].shape}")
print(f"revtq: {compustat[compustat["revtq"].isna()].shape}")
print(f"xrdq: {compustat[compustat["xrdq"].isna()].shape}")
print(compustat.shape)
## droping costat - only 2 options
compustat.drop(columns = ["costat"], inplace = True)
compustat.head()

invtq: (282876, 10)
oibdpq: (329753, 10)
ppentq: (287308, 10)
revtq: (296414, 10)
xrdq: (810316, 10)
(1215887, 10)


,gvkey,datadate,conm,datacqtr,invtq,oibdpq,ppentq,revtq,xrdq
0,1417,2016-06-30,INVESCO BOND FUND,2016Q2,NaN,NaN,NaN,NaN,NaN
1,1003,1990-01-31,A.A. IMPORTING CO INC,1989Q4,6.935,NaN,0.654,4.913,NaN
2,1003,1990-04-30,A.A. IMPORTING CO INC,1990Q1,6.652,NaN,0.591,3.506,NaN
3,1003,1990-07-31,A.A. IMPORTING CO INC,1990Q2,5.290,NaN,0.388,3.153,NaN
4,1003,1990-10-31,A.A. IMPORTING CO INC,1990Q3,5.044,NaN,0.346,2.678,NaN


To proceed with modelling the data, we need to ensure that the data is unique to the Year-Quarter, This is done by aggregating the data by "datacqtr".

After doing so, the growth rate of each needs to be calculated to match NGDP. This is because modelling on absolute data is not recommended for prediction.

In [4]:
compustat.columns

Index(['gvkey', 'datadate', 'conm', 'datacqtr', 'invtq', 'oibdpq', 'ppentq',
       'revtq', 'xrdq'],
      dtype='object')

In [5]:
## create aggregates for assets, liabilities and net income
agg_dict = {"revtq": "sum", "invtq": "sum", "oibdpq": "sum", "ppentq": "sum", "xrdq": "sum"}
compustat_macro = compustat.groupby("datacqtr").agg(agg_dict).reset_index()
compustat_macro.sort_values(by = "datacqtr", ascending = True, inplace = True)

for col in agg_dict.keys():
    ## calculating respective growth rates
    compustat_macro[f"{col}_growth"] = compustat_macro[col].pct_change()
    ## creating lags for forecasting
    compustat_macro[f"{col}_growth_lag"] = compustat_macro[f"{col}_growth"].shift(1)

## converting infinite values to NaN
compustat_macro.replace([np.inf, -np.inf], np.nan, inplace = True)
## dropping missing values
compustat_macro.dropna(inplace = True)
compustat_macro.head()

,datacqtr,revtq,invtq,oibdpq,ppentq,xrdq,revtq_growth,revtq_growth_lag,invtq_growth,invtq_growth_lag,oibdpq_growth,oibdpq_growth_lag,ppentq_growth,ppentq_growth_lag,xrdq_growth,xrdq_growth_lag
2,1990Q2,1598842.812,734508.760,253114.224,3051933.492,15453.975,-0.078831,11.756818,0.012967,8.670208,-0.028437,21.598241,-0.006882,29.912283,-0.280529,28.041547
3,1990Q3,1589997.845,759786.281,251447.114,3103095.216,18526.353,-0.005532,-0.078831,0.034414,0.012967,-0.006586,-0.028437,0.016764,-0.006882,0.198808,-0.280529
4,1990Q4,2002594.781,841510.020,287653.986,3334061.904,66747.693,0.259495,-0.005532,0.107561,0.034414,0.143994,-0.006586,0.074431,0.016764,2.602851,0.198808
5,1991Q1,1806337.148,773758.869,253396.124,3162845.621,25340.255,-0.098002,0.259495,-0.080511,0.107561,-0.119094,0.143994,-0.051354,0.074431,-0.620358,2.602851
6,1991Q2,1638951.532,764639.306,239511.504,3138964.250,16254.240,-0.092666,-0.098002,-0.011786,-0.080511,-0.054794,-0.119094,-0.007551,-0.051354,-0.358561,-0.620358


To join the data, since we are removing rows with missing values, we need to conduct inner join

In [6]:
# merge compustat data to train and test
train_comp_macro = train.merge(compustat_macro, how = "inner", left_on = "YQ", right_on = "datacqtr")
test_comp_macro = test.merge(compustat_macro, how = "inner", left_on = "YQ", right_on = "datacqtr")

Now 2 models will be done, to consider if modelling the data via growth would perform better, or modelling the data via growth_lag would perform better.

Before running both models, we will consider if keeping `qtr` is beneficial

In [7]:
formula1 = "NGDP ~ year + qtr + revtq_growth + invtq_growth + oibdpq_growth + ppentq_growth + xrdq_growth"
comp_model1 = smf.ols(formula1, train_comp_macro).fit()
print(comp_model1.summary())

y_pred = comp_model1.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
print(f"\nRMSE w/ Growths: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.206
Model:                            OLS   Adj. R-squared:                  0.129
Method:                 Least Squares   F-statistic:                     2.682
Date:                Wed, 04 Mar 2026   Prob (F-statistic):            0.00814
Time:                        17:59:05   Log-Likelihood:                -236.33
No. Observations:                 103   AIC:                             492.7
Df Residuals:                      93   BIC:                             519.0
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept       213.0559     74.669      2.853

In [8]:
print("-"*65 + "\n BASE MODEL vs GROWTH w/ QTR\n" + "-"*65)
calc_performance(formula0, base_model, formula1, comp_model1)
## Insights: adding accounting context slightly improves performance

-----------------------------------------------------------------
 BASE MODEL vs GROWTH w/ QTR
-----------------------------------------------------------------
                                               model   Adj.R^2
0                                  NGDP ~ year + qtr  0.082795
1  NGDP ~ year + qtr + revtq_growth + invtq_growt...  0.129208

   df_resid         ssr  df_diff    ss_diff  Chisq  Pr(>Chisq)
0      99.0  676.137650      0.0        NaN    0.0         0.0
1      93.0  593.369832      6.0  82.767818    0.0         0.0


In [9]:
formula2 = "NGDP ~ year + revtq_growth + invtq_growth + oibdpq_growth + ppentq_growth + xrdq_growth"
comp_model2 = smf.ols(formula2, train_comp_macro).fit()
print(comp_model2.summary())

y_pred = comp_model2.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred, "1_acct_growths_forecast")
print(f"\nRMSE w/ Growths (no qtr): {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.186
Model:                            OLS   Adj. R-squared:                  0.135
Method:                 Least Squares   F-statistic:                     3.647
Date:                Wed, 04 Mar 2026   Prob (F-statistic):            0.00264
Time:                        17:59:05   Log-Likelihood:                -237.64
No. Observations:                 103   AIC:                             489.3
Df Residuals:                      96   BIC:                             507.7
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept       197.1212     70.725      2.787

In [10]:
print("-"*65 + "\n GROWTH w/ QTR vs GROWTH w/o QTR\n" + "-"*65)
calc_performance(formula1, comp_model1, formula2, comp_model2)
## Insights: one less variable (qtr) had similar performance
# to prevent overfitting, qtr will be removed

-----------------------------------------------------------------
 GROWTH w/ QTR vs GROWTH w/o QTR
-----------------------------------------------------------------
                                               model   Adj.R^2
0  NGDP ~ year + qtr + revtq_growth + invtq_growt...  0.129208
1  NGDP ~ year + revtq_growth + invtq_growth + oi...  0.134744

   df_resid         ssr  df_diff    ss_diff  Chisq  Pr(>Chisq)
0      93.0  593.369832      0.0        NaN    0.0         0.0
1      96.0  608.616900     -3.0 -15.247068    0.0         0.0


With one less variable, the models performed relatively similar, while the model including `qtr` performed better mathematically, the model without is a better model as it achieves almost the same results with fewer variables, risking overfitting.

In [14]:
formula3 = "NGDP ~ year + revtq_growth_lag + invtq_growth_lag + oibdpq_growth_lag + ppentq_growth_lag + xrdq_growth_lag"
comp_model3 = smf.ols(formula3, train_comp_macro).fit()
print(comp_model3.summary())

y_pred = comp_model3.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred, "2_acct_growthlag_forecast")
print(f"\nRMSE w/ Growth Lags: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.226
Model:                            OLS   Adj. R-squared:                  0.177
Method:                 Least Squares   F-statistic:                     4.668
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           0.000331
Time:                        17:59:57   Log-Likelihood:                -235.03
No. Observations:                 103   AIC:                             484.1
Df Residuals:                      96   BIC:                             502.5
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           174.0474     69.46

In [15]:
print("-"*65 + "\n GROWTH vs GROWTH LAG\n" + "-"*65)
calc_performance(formula2, comp_model2, formula3, comp_model3)
# Insights: growth lag better models NGDP trends

-----------------------------------------------------------------
 GROWTH vs GROWTH LAG
-----------------------------------------------------------------
                                               model   Adj.R^2
0  NGDP ~ year + revtq_growth + invtq_growth + oi...  0.134744
1  NGDP ~ year + revtq_growth_lag + invtq_growth_...  0.177465

   df_resid         ssr  df_diff    ss_diff  Chisq  Pr(>Chisq)
0      96.0  608.616900      0.0        NaN    0.0         0.0
1      96.0  578.567158     -0.0  30.049742    0.0         0.0


A new model will be created with variables that have a p-value below 0.1

```                        coef    std err          t      P>|t|
oibdpq_growth         6.3310      3.547      1.785      0.077
xrdq_growth          -0.8825      0.492     -1.795      0.076
invtq_growth_lag     21.3640      6.103      3.500      0.001
ppentq_growth_lag    -5.6208      1.731     -3.246      0.002
xrdq_growth_lag      -1.2360      0.471     -2.622      0.010

In [16]:
formula4 = "NGDP ~ year + oibdpq_growth + xrdq_growth + invtq_growth_lag + ppentq_growth_lag + xrdq_growth_lag"
comp_model4 = smf.ols(formula4, train_comp_macro).fit()
print(comp_model4.summary())

y_pred = comp_model4.predict(test_comp_macro)
y_true = test_comp_macro["NGDP"]
create_submission(test_comp_macro, y_pred, "3_acct_combi_forecast")
print(f"\nRMSE Improved Acct Macro: {np.sqrt(mean_squared_error(y_true, y_pred))}")

                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.317
Model:                            OLS   Adj. R-squared:                  0.274
Method:                 Least Squares   F-statistic:                     7.416
Date:                Wed, 04 Mar 2026   Prob (F-statistic):           1.54e-06
Time:                        18:09:32   Log-Likelihood:                -228.60
No. Observations:                 103   AIC:                             471.2
Df Residuals:                      96   BIC:                             489.6
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           198.4387     65.95

In [18]:
print("-"*65 + "\n ALL GROWTH LAG vs COMBINATION MODEL\n" + "-"*65)
calc_performance(formula3, comp_model3, formula4, comp_model4)
## Insights: combining variables that have higher significance to forecasting NGDP improved the model
## however, this did not translate to a better performance on Kaggle

-----------------------------------------------------------------
 ALL GROWTH LAG vs COMBINATION MODEL
-----------------------------------------------------------------
                                               model   Adj.R^2
0  NGDP ~ year + revtq_growth_lag + invtq_growth_...  0.177465
1  NGDP ~ year + oibdpq_growth + xrdq_growth + in...  0.274003

   df_resid         ssr  df_diff    ss_diff  Chisq  Pr(>Chisq)
0      96.0  578.567158      0.0        NaN    0.0         0.0
1      96.0  510.662783     -0.0  67.904375    0.0         0.0


We recognised from this that:
1. `qtr` can be removed to prevent risk of overfitting as there is no significant change in performance from it
2. while the model has improved mathematically, the performance on Kaggle has worsened

### Attempting other models with same formula
With this im mind, we will proceed to attempt other models with the same formula:<br>
`formula4 = "NGDP ~ year + oibdpq_growth + xrdq_growth + invtq_growth_lag + ppentq_growth_lag + xrdq_growth_lag"`

This is to ensure that our insights focus only on the model choice performance, instead of the variable selection

### How Stock Price / Return Data affects GDP (CRSP on WRDS)
1. Essential Variables (The "Core" Signal)
- (dlycaldt) Daily Calendar Date: daily moves into Year-Quarter (YQ) buckets
- (dlytotret) Daily Total Return on Index: This includes dividends. In forensic finance, this is the most powerful leading indicator. It represents the market's "real-time" discount of future economic growth.
- (dlytotval) Index Total Capitalization Value: This is the aggregate dollar value of all firms in the index. Use this to measure the Wealth Effect. If this value rises significantly, consumer spending (the biggest part of GDP) usually follows 1–2 quarters later.

2. Forensic Variables (To Lower Your RMSE)
- (dlytotind) Daily Index Level: Useful if you want to calculate the "momentum" of the market (e.g., comparing the current level to a 200-day moving average).
- (dlyincret) Daily Income Return on Index: This isolates the dividend component. A high income return relative to price return often signals a "flight to quality," which forensically precedes economic slowdowns.
- (dlyusdcnt) Used Count: The number of firms currently in the index. Rapid changes in the count of firms can signal structural shifts in the economy (like a wave of IPOs or bankruptcies).

### How Analysts' forecast affects GDP (Survey of Professional Forecasters)